# Multi-Seed Ablation Re-Run untuk Validasi Statistik

Notebook ini menjalankan ulang 8 konfigurasi ablation sebanyak 5x dengan seed berbeda.
Output-nya akan digunakan untuk mengisi **Table 8** di paper (mean ± std + Wilcoxon test).

**PENTING:** Pastikan `best_params` dari Optuna sudah tersedia sebelum menjalankan notebook ini.
Isi manual di Cell 2 jika Optuna tidak dijalankan ulang.

In [1]:
# ============================================================
# CELL 1 — Install & Import
# ============================================================
!pip install optuna scipy openpyxl -q

import pandas as pd
import numpy as np
import re
import random
import os
import warnings
warnings.filterwarnings('ignore')

import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from transformers import XLMRobertaTokenizer, XLMRobertaModel
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import f1_score, accuracy_score, roc_auc_score
from scipy.stats import wilcoxon

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')


Using device: cuda


In [2]:
# ============================================================
# CELL 2 — BEST PARAMS dari Optuna (ISI MANUAL JIKA PERLU)
# ============================================================
# Salin nilai best_params dari hasil Optuna Anda.
# Jika study masih tersedia di memory, ganti saja dengan:
# best_params = study.best_params

best_params = {
   # 'learning_rate'       : 2e-5,    # <-- ganti dengan nilai Anda
    #'batch_size'          : 32,      # <-- ganti dengan nilai Anda
    #'max_length'          : 128,     # <-- ganti dengan nilai Anda
    #'tfidf_features'      : 1000,    # <-- ganti dengan nilai Anda
    #'tfidf_hidden_dim'    : 128,     # <-- ganti dengan nilai Anda
    #'stat_hidden_dim'     : 16,      # <-- ganti dengan nilai Anda
    #'classifier_hidden_dim': 128,    # <-- ganti dengan nilai Anda
    #'dropout_rate'        : 0.3,     # <-- ganti dengan nilai Anda
    #'weight_decay'        : 1e-4,    # <-- ganti dengan nilai Anda
    'learning_rate'  : 4.6868489058130973e-05,
    'batch_size'  : 32,
    'max_length'  : 128,
    'tfidf_features'  : 1750,
    'tfidf_hidden_dim'  : 64,
    'stat_hidden_dim'  : 32,
    'classifier_hidden_dim'  : 224,
    'dropout_rate'  : 0.3816095435279173,
    'weight_decay'  : 0.00014970099966199086,
}

# Seeds yang akan digunakan
SEEDS      = [42, 123, 456, 789, 2025]
NUM_EPOCHS = 50   # sama dengan training final di notebook asli
PATIENCE   = 10   # early stopping

print('Best params loaded:', best_params)

Best params loaded: {'learning_rate': 4.6868489058130973e-05, 'batch_size': 32, 'max_length': 128, 'tfidf_features': 1750, 'tfidf_hidden_dim': 64, 'stat_hidden_dim': 32, 'classifier_hidden_dim': 224, 'dropout_rate': 0.3816095435279173, 'weight_decay': 0.00014970099966199086}


In [3]:
# ============================================================
# CELL 3 — Load Data
# ============================================================
train_df = pd.read_excel('train_data_with_codemixed_preprocessing.xlsx')
val_df   = pd.read_excel('val_data_with_codemixed_preprocessing.xlsx')
test_df  = pd.read_excel('test_data_with_codemixed_preprocessing.xlsx')

print(f'Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}')

Train: 8752 | Val: 2189 | Test: 2736


In [4]:
# ============================================================
# CELL 4 — Utility: set_seed
# ============================================================
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark     = False

In [5]:
# ============================================================
# CELL 5 — Feature Preparation (sama persis dengan notebook asli)
# ============================================================
def prepare_features(train_df, val_df, test_df, max_features=1000):
    tfidf = TfidfVectorizer(max_features=max_features)
    X_tfidf_train = tfidf.fit_transform(train_df['tweet_clean_codemixed']).toarray()
    X_tfidf_val   = tfidf.transform(val_df['tweet_clean_codemixed']).toarray()
    X_tfidf_test  = tfidf.transform(test_df['tweet_clean_codemixed']).toarray()

    def extract_stat_features(lang_seq):
        id_count  = len(re.findall(r'\[id\]',  lang_seq))
        en_count  = len(re.findall(r'\[en\]',  lang_seq))
        unk_count = len(re.findall(r'\[\?\?\]', lang_seq))
        total = id_count + en_count + unk_count
        return [total,
                id_count  / total if total else 0,
                en_count  / total if total else 0,
                unk_count / total if total else 0]

    X_stat_train = np.array([extract_stat_features(t) for t in train_df['lang_per_word']])
    X_stat_val   = np.array([extract_stat_features(t) for t in val_df['lang_per_word']])
    X_stat_test  = np.array([extract_stat_features(t) for t in test_df['lang_per_word']])

    scaler = StandardScaler()
    X_stat_train = scaler.fit_transform(X_stat_train)
    X_stat_val   = scaler.transform(X_stat_val)
    X_stat_test  = scaler.transform(X_stat_test)

    return (X_tfidf_train, X_tfidf_val, X_tfidf_test,
            X_stat_train,  X_stat_val,  X_stat_test,
            tfidf, scaler)

In [6]:
# ============================================================
# CELL 6 — Dataset Class (sama persis dengan notebook asli)
# ============================================================
tokenizer = XLMRobertaTokenizer.from_pretrained('xlm-roberta-base')

class HybridDataset(Dataset):
    def __init__(self, texts, tfidf, stat, labels, max_length=128):
        self.encodings = tokenizer(
            texts.tolist(), padding=True, truncation=True,
            max_length=max_length, return_tensors='pt'
        )
        self.tfidf  = torch.tensor(tfidf,  dtype=torch.float32)
        self.stat   = torch.tensor(stat,   dtype=torch.float32)
        self.labels = torch.tensor(labels.tolist(), dtype=torch.long)

    def __getitem__(self, idx):
        item = {k: v[idx] for k, v in self.encodings.items()}
        item['tfidf']  = self.tfidf[idx]
        item['stat']   = self.stat[idx]
        item['labels'] = self.labels[idx]
        return item

    def __len__(self):
        return len(self.labels)

In [7]:
# ============================================================
# CELL 7 — Model yang Dapat Dikonfigurasi untuk Ablation
#
# use_tfidf  : apakah menggunakan TF-IDF branch
# use_stat   : apakah menggunakan Statistical feature branch
# freeze_layers: apakah membekukan 6 layer pertama XLM-RoBERTa
# ============================================================
class AblationModel(nn.Module):
    def __init__(self, params, use_tfidf=True, use_stat=True, freeze_layers=True):
        super().__init__()
        self.use_tfidf = use_tfidf
        self.use_stat  = use_stat

        self.xlm = XLMRobertaModel.from_pretrained('xlm-roberta-base')

        # Layer freezing strategy
        if freeze_layers:
            for param in self.xlm.embeddings.parameters():
                param.requires_grad = False
            for layer in self.xlm.encoder.layer[:6]:
                for param in layer.parameters():
                    param.requires_grad = False

        # Ukuran input classifier
        combined_dim = 768

        if use_tfidf:
            self.tfidf_layer = nn.Sequential(
                nn.Linear(params['tfidf_features'], params['tfidf_hidden_dim']),
                nn.ReLU(),
                nn.Dropout(params['dropout_rate'])
            )
            combined_dim += params['tfidf_hidden_dim']

        if use_stat:
            self.stat_layer = nn.Sequential(
                nn.Linear(4, params['stat_hidden_dim']),
                nn.ReLU(),
                nn.Dropout(params['dropout_rate'])
            )
            combined_dim += params['stat_hidden_dim']

        self.classifier = nn.Sequential(
            nn.Linear(combined_dim, params['classifier_hidden_dim']),
            nn.ReLU(),
            nn.Dropout(params['dropout_rate']),
            nn.Linear(params['classifier_hidden_dim'], 2)
        )

    def forward(self, input_ids, attention_mask, tfidf, stat):
        xlm_out = self.xlm(
            input_ids=input_ids,
            attention_mask=attention_mask
        ).last_hidden_state[:, 0, :]

        parts = [xlm_out]
        if self.use_tfidf:
            parts.append(self.tfidf_layer(tfidf))
        if self.use_stat:
            parts.append(self.stat_layer(stat))

        combined = torch.cat(parts, dim=1)
        return self.classifier(combined)

In [8]:
# ============================================================
# CELL 8 — Definisi 8 Konfigurasi Ablation
# ============================================================
ABLATION_CONFIGS = {
    # Nama config           : (use_tfidf, use_stat, freeze_layers)
    'baseline_full'         : (True,  True,  True ),   # F1=0.9264 (paper)
    'no_tfidf'              : (False, True,  True ),   # F1=0.9175
    'no_stat_features'      : (True,  False, True ),   # F1=0.9175
    'no_stat_no_tfidf'      : (False, False, True ),   # F1=0.9194
    'full_no_freezing'      : (True,  True,  False),   # F1=0.9309
    'no_tfidf_no_freezing'  : (False, True,  False),   # F1=0.9282
    'no_stat_no_freezing'   : (False, False, False),   # F1=0.9371 ← BEST
    'no_stat_feat_no_freeze': (True,  False, False),   # F1=0.9282
}

print('8 konfigurasi ablation siap:')
for name, (tf, st, fr) in ABLATION_CONFIGS.items():
    print(f'  {name:<28} | TF-IDF={tf} | Stat={st} | Freeze={fr}')

8 konfigurasi ablation siap:
  baseline_full                | TF-IDF=True | Stat=True | Freeze=True
  no_tfidf                     | TF-IDF=False | Stat=True | Freeze=True
  no_stat_features             | TF-IDF=True | Stat=False | Freeze=True
  no_stat_no_tfidf             | TF-IDF=False | Stat=False | Freeze=True
  full_no_freezing             | TF-IDF=True | Stat=True | Freeze=False
  no_tfidf_no_freezing         | TF-IDF=False | Stat=True | Freeze=False
  no_stat_no_freezing          | TF-IDF=False | Stat=False | Freeze=False
  no_stat_feat_no_freeze       | TF-IDF=True | Stat=False | Freeze=False


In [9]:
# ============================================================
# CELL 9 — Fungsi Single Training Run
# ============================================================
def run_single(seed, config_name, params,
               X_tfidf_train, X_tfidf_val, X_tfidf_test,
               X_stat_train,  X_stat_val,  X_stat_test):

    set_seed(seed)
    use_tfidf, use_stat, freeze_layers = ABLATION_CONFIGS[config_name]

    # Dataset
    tr = HybridDataset(train_df['tweet_for_model'], X_tfidf_train, X_stat_train,
                       train_df['label'], max_length=params['max_length'])
    va = HybridDataset(val_df['tweet_for_model'],   X_tfidf_val,   X_stat_val,
                       val_df['label'],   max_length=params['max_length'])
    te = HybridDataset(test_df['tweet_for_model'],  X_tfidf_test,  X_stat_test,
                       test_df['label'],  max_length=params['max_length'])

    tr_loader = DataLoader(tr, batch_size=params['batch_size'], shuffle=True)
    va_loader = DataLoader(va, batch_size=params['batch_size'])
    te_loader = DataLoader(te, batch_size=params['batch_size'])

    # Model
    model = AblationModel(
        params, use_tfidf=use_tfidf,
        use_stat=use_stat, freeze_layers=freeze_layers
    ).to(device)

    optimizer = AdamW(model.parameters(),
                      lr=params['learning_rate'],
                      weight_decay=params['weight_decay'])
    loss_fn   = nn.CrossEntropyLoss()

    best_val_loss = float('inf')
    patience_cnt  = 0
    best_state    = None

    for epoch in range(NUM_EPOCHS):
        # --- Train ---
        model.train()
        for batch in tr_loader:
            ids  = batch['input_ids'].to(device)
            mask = batch['attention_mask'].to(device)
            tf   = batch['tfidf'].to(device)
            st   = batch['stat'].to(device)
            lbl  = batch['labels'].to(device)

            optimizer.zero_grad()
            out  = model(ids, mask, tf, st)
            loss = loss_fn(out, lbl)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()

        # --- Validate ---
        model.eval()
        val_loss = 0
        with torch.no_grad():
            for batch in va_loader:
                ids  = batch['input_ids'].to(device)
                mask = batch['attention_mask'].to(device)
                tf   = batch['tfidf'].to(device)
                st   = batch['stat'].to(device)
                lbl  = batch['labels'].to(device)
                out  = model(ids, mask, tf, st)
                val_loss += loss_fn(out, lbl).item()

        val_loss /= len(va_loader)
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_state    = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            patience_cnt  = 0
        else:
            patience_cnt += 1
            if patience_cnt >= PATIENCE:
                break

    # --- Test dengan best model ---
    model.load_state_dict({k: v.to(device) for k, v in best_state.items()})
    model.eval()

    all_preds, all_labels, all_probs = [], [], []
    with torch.no_grad():
        for batch in te_loader:
            ids  = batch['input_ids'].to(device)
            mask = batch['attention_mask'].to(device)
            tf   = batch['tfidf'].to(device)
            st   = batch['stat'].to(device)
            lbl  = batch['labels'].to(device)

            out   = model(ids, mask, tf, st)
            prob  = torch.softmax(out, dim=1)[:, 1].cpu().numpy()
            pred  = torch.argmax(out, dim=1).cpu().numpy()

            all_preds.extend(pred)
            all_labels.extend(lbl.cpu().numpy())
            all_probs.extend(prob)

    f1  = f1_score(all_labels, all_preds, average='weighted')
    acc = accuracy_score(all_labels, all_preds)
    auc = roc_auc_score(all_labels, all_probs)

    return f1, acc, auc

In [10]:
# ============================================================
# CELL 10 — MAIN LOOP: 8 Config × 5 Seeds
# Estimasi waktu: 8 × 5 × ~47 menit = ±31 jam
# Jalankan malam hari atau gunakan GPU cloud
# ============================================================

# Siapkan fitur sekali saja untuk efisiensi
(X_tfidf_train, X_tfidf_val, X_tfidf_test,
 X_stat_train,  X_stat_val,  X_stat_test,
 _, _) = prepare_features(
    train_df, val_df, test_df,
    max_features=best_params['tfidf_features']
)

# Tempat menyimpan hasil
all_results = {
    cfg: {'f1': [], 'acc': [], 'auc': []}
    for cfg in ABLATION_CONFIGS
}

total_runs = len(ABLATION_CONFIGS) * len(SEEDS)
run_counter = 0

for cfg_name in ABLATION_CONFIGS:
    print("\n" + "=" * 60)
    print(f"Config: {cfg_name}")
    print("=" * 60)

    for seed in SEEDS:
        run_counter += 1

        print(
            f"  [{run_counter}/{total_runs}] Seed {seed}...",
            end=" ",
            flush=True
        )

        f1, acc, auc = run_single(
            seed, cfg_name, best_params,
            X_tfidf_train, X_tfidf_val, X_tfidf_test,
            X_stat_train,  X_stat_val,  X_stat_test
        )

        all_results[cfg_name]['f1'].append(f1)
        all_results[cfg_name]['acc'].append(acc)
        all_results[cfg_name]['auc'].append(auc)

        print(f"F1={f1:.4f} | Acc={acc:.4f} | AUC={auc:.4f}")

print("\n✅ Semua run selesai!")


Config: baseline_full
  [1/40] Seed 42... F1=0.9346 | Acc=0.9346 | AUC=0.9836
  [2/40] Seed 123... F1=0.9298 | Acc=0.9298 | AUC=0.9761
  [3/40] Seed 456... F1=0.9206 | Acc=0.9207 | AUC=0.9798
  [4/40] Seed 789... F1=0.9364 | Acc=0.9364 | AUC=0.9832
  [5/40] Seed 2025... F1=0.9364 | Acc=0.9364 | AUC=0.9826

Config: no_tfidf
  [6/40] Seed 42... F1=0.9313 | Acc=0.9313 | AUC=0.9819
  [7/40] Seed 123... F1=0.9320 | Acc=0.9320 | AUC=0.9826
  [8/40] Seed 456... F1=0.9279 | Acc=0.9280 | AUC=0.9792
  [9/40] Seed 789... F1=0.9338 | Acc=0.9338 | AUC=0.9816
  [10/40] Seed 2025... F1=0.9401 | Acc=0.9401 | AUC=0.9829

Config: no_stat_features
  [11/40] Seed 42... F1=0.9229 | Acc=0.9229 | AUC=0.9787
  [12/40] Seed 123... F1=0.9327 | Acc=0.9327 | AUC=0.9821
  [13/40] Seed 456... F1=0.9301 | Acc=0.9302 | AUC=0.9803
  [14/40] Seed 789... F1=0.9360 | Acc=0.9360 | AUC=0.9805
  [15/40] Seed 2025... F1=0.9338 | Acc=0.9338 | AUC=0.9814

Config: no_stat_no_tfidf
  [16/40] Seed 42... F1=0.9389 | Acc=0.9390 | 

In [11]:
# ============================================================
# CELL 11 — Hitung Mean ± Std
# ============================================================
summary_rows = []

print(f'\n{"Configuration":<28} | {"F1 (mean±std)":<18} | {"Acc (mean±std)":<18} | {"AUC (mean±std)"}')
print('-' * 90)

for cfg, res in all_results.items():
    f1_arr  = np.array(res['f1'])
    acc_arr = np.array(res['acc'])
    auc_arr = np.array(res['auc'])

    row = {
        'Configuration' : cfg,
        'F1_mean'  : f1_arr.mean(),
        'F1_std'   : f1_arr.std(),
        'Acc_mean' : acc_arr.mean(),
        'Acc_std'  : acc_arr.std(),
        'AUC_mean' : auc_arr.mean(),
        'AUC_std'  : auc_arr.std(),
        'F1_all'   : res['f1'],
    }
    summary_rows.append(row)

    print(f'{cfg:<28} | '
          f'{f1_arr.mean():.4f}±{f1_arr.std():.4f}   | '
          f'{acc_arr.mean():.4f}±{acc_arr.std():.4f}   | '
          f'{auc_arr.mean():.4f}±{auc_arr.std():.4f}')

summary_df = pd.DataFrame(summary_rows)


Configuration                | F1 (mean±std)      | Acc (mean±std)     | AUC (mean±std)
------------------------------------------------------------------------------------------
baseline_full                | 0.9316±0.0060   | 0.9316±0.0060   | 0.9811±0.0028
no_tfidf                     | 0.9330±0.0040   | 0.9330±0.0040   | 0.9816±0.0013
no_stat_features             | 0.9311±0.0045   | 0.9311±0.0045   | 0.9806±0.0012
no_stat_no_tfidf             | 0.9351±0.0026   | 0.9351±0.0026   | 0.9815±0.0012
full_no_freezing             | 0.9312±0.0044   | 0.9312±0.0044   | 0.9800±0.0008
no_tfidf_no_freezing         | 0.9277±0.0036   | 0.9278±0.0036   | 0.9784±0.0018
no_stat_no_freezing          | 0.9290±0.0024   | 0.9290±0.0024   | 0.9790±0.0019
no_stat_feat_no_freeze       | 0.9272±0.0030   | 0.9272±0.0030   | 0.9772±0.0021


In [12]:
# ============================================================
# CELL 12 — Wilcoxon Signed-Rank Test
# Membandingkan setiap konfigurasi dengan baseline_full
# ============================================================
from scipy.stats import wilcoxon

baseline_f1 = np.array(all_results['baseline_full']['f1'])
best_f1     = np.array(all_results['no_stat_no_freezing']['f1'])

wilcoxon_rows = []
print(f'\n{"Configuration":<28} | p-value  | Signifikan (p<0.05)?')
print('-' * 65)

for cfg, res in all_results.items():
    if cfg == 'baseline_full':
        wilcoxon_rows.append({'Configuration': cfg, 'p_value': '—', 'significant': '—'})
        print(f'{cfg:<28} | —        | — (baseline)')
        continue

    cfg_f1 = np.array(res['f1'])

    # Cek apakah semua nilai identik (wilcoxon butuh variasi)
    if np.all(cfg_f1 == baseline_f1):
        p = 1.0
    else:
        try:
            # alternative='greater': apakah config ini > baseline?
            _, p = wilcoxon(cfg_f1, baseline_f1, alternative='greater')
        except ValueError:
            # Jika tidak ada perbedaan (zero differences)
            _, p = wilcoxon(cfg_f1, baseline_f1)

    sig = '✅ Yes' if p < 0.05 else '❌ No'
    wilcoxon_rows.append({'Configuration': cfg, 'p_value': round(p, 4), 'significant': sig})
    print(f'{cfg:<28} | {p:.4f}   | {sig}')

wilcoxon_df = pd.DataFrame(wilcoxon_rows)

# Khusus no_stat_no_freezing vs baseline_full (klaim utama paper)
print(f'\n--- Uji Klaim Utama Paper ---')
print(f'no_stat_no_freezing vs baseline_full:')
if not np.all(best_f1 == baseline_f1):
    stat, p_main = wilcoxon(best_f1, baseline_f1, alternative='greater')
    print(f'  Statistic : {stat:.4f}')
    print(f'  p-value   : {p_main:.4f}')
    print(f'  Hasil     : {"SIGNIFIKAN ✅" if p_main < 0.05 else "Tidak Signifikan ❌"}')


Configuration                | p-value  | Signifikan (p<0.05)?
-----------------------------------------------------------------
baseline_full                | —        | — (baseline)
no_tfidf                     | 0.3125   | ❌ No
no_stat_features             | 0.5938   | ❌ No
no_stat_no_tfidf             | 0.1562   | ❌ No
full_no_freezing             | 0.6875   | ❌ No
no_tfidf_no_freezing         | 0.8438   | ❌ No
no_stat_no_freezing          | 0.8438   | ❌ No
no_stat_feat_no_freeze       | 0.7812   | ❌ No

--- Uji Klaim Utama Paper ---
no_stat_no_freezing vs baseline_full:
  Statistic : 4.0000
  p-value   : 0.8438
  Hasil     : Tidak Signifikan ❌


In [13]:
# ============================================================
# CELL 13 — Simpan Hasil ke Excel (untuk dimasukkan ke Table 8)
# ============================================================

# Tabel ringkasan (mean ± std)
table8_rows = []
for cfg, res in all_results.items():
    f1_arr  = np.array(res['f1'])
    acc_arr = np.array(res['acc'])
    auc_arr = np.array(res['auc'])

    # Cari p-value dari wilcoxon_df
    pval_row = wilcoxon_df[wilcoxon_df['Configuration'] == cfg]
    pval = pval_row['p_value'].values[0] if len(pval_row) > 0 else '—'
    sig  = pval_row['significant'].values[0] if len(pval_row) > 0 else '—'

    table8_rows.append({
        'Configuration'      : cfg,
        'F1 (mean ± std)'    : f'{f1_arr.mean():.4f} ± {f1_arr.std():.4f}',
        'Acc (mean ± std)'   : f'{acc_arr.mean():.4f} ± {acc_arr.std():.4f}',
        'AUC (mean ± std)'   : f'{auc_arr.mean():.4f} ± {auc_arr.std():.4f}',
        'p-value vs baseline': pval,
        'Significant?'       : sig,
    })

table8_df = pd.DataFrame(table8_rows)

# Simpan semua ke satu file Excel
with pd.ExcelWriter('multi_seed_results.xlsx', engine='openpyxl') as writer:
    table8_df.to_excel(writer, sheet_name='Table8_Summary', index=False)
    wilcoxon_df.to_excel(writer, sheet_name='Wilcoxon_Details', index=False)
    # Raw results per seed
    raw_rows = []
    for cfg, res in all_results.items():
        for i, seed in enumerate(SEEDS):
            raw_rows.append({
                'Configuration': cfg, 'Seed': seed,
                'F1': res['f1'][i], 'Acc': res['acc'][i], 'AUC': res['auc'][i]
            })
    pd.DataFrame(raw_rows).to_excel(writer, sheet_name='Raw_Per_Seed', index=False)

print('✅ Hasil disimpan ke: multi_seed_results.xlsx')
print('\n📋 Preview Table 8:')
print(table8_df.to_string(index=False))

✅ Hasil disimpan ke: multi_seed_results.xlsx

📋 Preview Table 8:
         Configuration F1 (mean ± std) Acc (mean ± std) AUC (mean ± std) p-value vs baseline Significant?
         baseline_full 0.9316 ± 0.0060  0.9316 ± 0.0060  0.9811 ± 0.0028                   —            —
              no_tfidf 0.9330 ± 0.0040  0.9330 ± 0.0040  0.9816 ± 0.0013              0.3125         ❌ No
      no_stat_features 0.9311 ± 0.0045  0.9311 ± 0.0045  0.9806 ± 0.0012              0.5938         ❌ No
      no_stat_no_tfidf 0.9351 ± 0.0026  0.9351 ± 0.0026  0.9815 ± 0.0012              0.1562         ❌ No
      full_no_freezing 0.9312 ± 0.0044  0.9312 ± 0.0044  0.9800 ± 0.0008              0.6875         ❌ No
  no_tfidf_no_freezing 0.9277 ± 0.0036  0.9278 ± 0.0036  0.9784 ± 0.0018              0.8438         ❌ No
   no_stat_no_freezing 0.9290 ± 0.0024  0.9290 ± 0.0024  0.9790 ± 0.0019              0.8438         ❌ No
no_stat_feat_no_freeze 0.9272 ± 0.0030  0.9272 ± 0.0030  0.9772 ± 0.0021              0

## Langkah Selanjutnya

Setelah notebook ini selesai berjalan:

1. **Download** file `multi_seed_results.xlsx`
2. **Upload ke Claude** — Claude akan membaca tabel tersebut dan otomatis menyisipkan Table 8 ke dalam file `.docx` revisi
3. Narasi di bagian **Discussion** dan **Limitations** juga akan diperbarui sesuai hasil nyata

Jika p-value **< 0.05** untuk `no_stat_no_freezing vs baseline_full`, klaim utama paper terkonfirmasi secara statistik.